# 08_11_ToTと_GoTの骨組み

コード：`ch08/blocks/08_11_ToTと_GoTの骨組み.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 8 章 12 節：ToT（木で探す）と GoT（枝を合流させる）の骨組み。LLM の役割（候補を出す・評価する）は関数に置き換える
import itertools, random
from fractions import Fraction

# ---- ToT：4 つの数と四則演算で 24 を作る（ToT の原論文の課題の 1 つ） ----------------
def propose(nums):                       # 「次の一手」の候補を全部出す（LLM なら「候補を出させる」呼び出し）
    out = []
    for i, j in itertools.permutations(range(len(nums)), 2):
        a, b = nums[i], nums[j]
        rest = [n for k, n in enumerate(nums) if k not in (i, j)]
        for v in (a + b, a - b, a * b) + ((a / b,) if b != 0 else ()):
            out.append(tuple(sorted(rest + [v])))
    return list(set(out))

def evaluate(nums):                      # 「この状態から 24 に届きそうか」の点数（LLM なら「評価させる」呼び出し）
    if len(nums) == 1:
        return 1.0 if nums[0] == 24 else 0.0
    best = min(abs(v - 24) for s in propose(nums) for v in s)   # 1 手先で 24 にどれだけ近づけるか
    return 1 / (1 + float(best))

def tot(nums, beam):                     # 幅 beam の幅優先探索。各段で点数の高い beam 個だけ残す
    frontier, calls = [tuple(Fraction(n) for n in nums)], 0
    for _ in range(3):
        cands = {c for s in frontier for c in propose(list(s))}
        calls += len(cands)
        frontier = sorted(cands, key=lambda c: -evaluate(list(c)))[:beam]
    return any(s[0] == 24 for s in frontier), calls

rng = random.Random(0)
puzzles = []
while len(puzzles) < 30:                 # 解ける問題だけを 30 問集める
    p = [rng.randint(1, 13) for _ in range(4)]
    if tot(p, beam=10_000)[0]:
        puzzles.append(p)
for beam in (1, 3, 5, 10_000):
    res = [tot(p, beam) for p in puzzles]
    label = "全探索" if beam == 10_000 else f"幅 {beam}"
    print(f"ToT {label:6s} 解けた {sum(ok for ok, _ in res):2d}/30  評価の呼び出し（1 問あたり） {sum(c for _, c in res) / 30:6.0f}")

# ---- GoT：ときどき間違える「並べ替え役」で 32 個の数を並べる（GoT の原論文の例の 1 つ） --------
def noisy_sort(xs, rng, p=0.04):         # 並べ替えるが、隣どうしを確率 p で入れ替えてしまう（LLM の写し間違いの代わり）
    ys = sorted(xs)
    for i in range(len(ys) - 1):
        if rng.random() < p:
            ys[i], ys[i + 1] = ys[i + 1], ys[i]
    return ys

def errors(ys):                          # 正しく並んでいない隣どうしの数（評価役。LLM なら「採点させる」）
    return sum(a > b for a, b in zip(ys, ys[1:]))

def noisy_merge(a, b, rng):              # 2 つの並んだ列を 1 つにまとめる（集約。これも少し間違える）
    return noisy_sort(a + b, rng, p=0.01)

def got(xs, rng, tries=3):               # 分ける → 各部分を 3 回ずつ並べ、評価で一番良いものを選ぶ → 合流させる
    parts, calls = [xs[i:i + 8] for i in range(0, len(xs), 8)], 0
    best = [min((noisy_sort(p, rng) for _ in range(tries)), key=errors) for p in parts]
    calls += len(parts) * tries
    while len(best) > 1:
        best = [min((noisy_merge(best[i], best[i + 1], rng) for _ in range(tries)), key=errors)
                for i in range(0, len(best), 2)]
        calls += len(best) * tries
    return best[0], calls

rng = random.Random(0)
data = [rng.sample(range(1000), 32) for _ in range(200)]
cot_err = sum(errors(noisy_sort(x, rng)) for x in data) / len(data)
res = [got(x, rng) for x in data]
got_err = sum(errors(y) for y, _ in res) / len(data)
print(f"並べ替え（32 個 × 200 回）：一度に並べる 平均誤り {cot_err:.2f}（並べ替え役の呼び出し 1 回）")
print(f"                         GoT（分割・選択・合流） 平均誤り {got_err:.2f}（呼び出し {res[0][1]} 回）")
